# Build Customer Metrics
Aggregates customer and order data from the Silver Layer to create customer-level metrics for analysis.

In [0]:
CREATE OR REPLACE TABLE ecommerce_catalog.gold.customer_metrics AS
SELECT 
    c.customer_id,
    c.first_name,
    c.last_name,
    c.country,
    c.customer_status,

    
    -- Kennzahlen
    COUNT(o.customer_id)                                AS order_count,
    COALESCE(SUM(o.quantity), 0)                        AS total_units,
    
    -- Gerundeter Gesamtumsatz
    ROUND(
        COALESCE(
            SUM(o.quantity * o.unit_price), 0.0
        ), 
        2
    ) AS total_revenue,
    
    -- Durchschnittlicher Umsatz pro Bestellung
    ROUND(
        COALESCE(
            SUM(o.quantity * o.unit_price) / NULLIF(COUNT(o.order_id), 0), 
            0.0
        ), 
        2
    ) AS avg_revenue_per_order,
    MIN(o.order_date)                                   AS first_order_date,
    MAX(o.order_date)                                   AS last_order_date
    
FROM ecommerce_catalog.silver.silver_customers c
LEFT JOIN ecommerce_catalog.silver.silver_orders o 
    ON c.customer_id = o.customer_id
GROUP BY 
    c.customer_id,
    c.first_name,
    c.last_name,
    c.country,
    c.customer_status
ORDER BY 
    total_revenue DESC;